**Without Pydantic → Python doesn't automatically enforce type hints.**

In [1]:
# before pydantic 

class Student:
    def __init__(self, name: str, age: int, gpa: float):
        self.name = name 
        self.age = age 
        self.gpa = gpa 

s = Student(name = 99, age = "Twenty", gpa = False)
print(s.name, " ", s.age, " ", s.gpa)

99   Twenty   False


**With Pydantic → data is validated when the model is created.**

In [2]:
# after pydantic 

from pydantic import BaseModel 

class Student(BaseModel):
    name:str 
    age:int
    gpa:float

s1 = Student(name = "Ram", age=21, gpa=2.9)
print(s1)

name='Ram' age=21 gpa=2.9


In [ ]:
s2 =  Student(name = 99, age = "Twenty", gpa = "excellent")
print(s2)

`ValidationError`

Pydantic found **3 validation errors**:

- **name:** Expected string, received integer (`99`).
- **age:** Expected integer, received `"Twenty"`.
- **gpa:** Expected float, received `"excellent"`.

**Conclusion:** Pydantic raises `ValidationError` when input data has invalid types or values.


`Real world approach, in API data will arrives in Dictionary format. `


> **Validation & Coercion**

Pydantic commonly attempts to convert compatible input types to the expected type.

In [ ]:
raw_data = {
    "name":"Sham",
    "age":20,
    "gpa":3.5
}

s2 = Student(**raw_data)    #** unpacks a dictionary into keyword arguments.
print(s2)

name='Sham' age=20 gpa=3.5


In [5]:
raw_data1 = {
    "name":"Sham",
    "age":"21",
    "gpa":"21.5"
}

s3 = Student(**raw_data1)
print(s3)
print(type(s3.name), type(s3.age), type(s3.gpa))

name='Sham' age=21 gpa=21.5
<class 'str'> <class 'int'> <class 'float'>


In [ ]:
raw_data2 = {
    "name":"Sham",
    "age":21.5,
    "gpa":"21.5"
}

s3 = Student(**raw_data2)
print(s3)

`ValidationError`

Pydantic found **1 validation error**:

- **age:** Expected an integer, but received `21.5` (float).

**Reason:** Pydantic cannot convert a fractional number into an integer automatically.


> #### **ValidationError** 

In [7]:
from pydantic import ValidationError 

raw_data4 = {
    "name" : 1234, 
    "age":"Twenty"
}

try:
    s4 = Student(**raw_data4)
except ValidationError as e: 
    print(f"Total number of errors found: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of errors found: 3
Fields : ('name',)  , Type : string_type ,  Message: Input should be a valid string
Fields : ('age',)  , Type : int_parsing ,  Message: Input should be a valid integer, unable to parse string as an integer
Fields : ('gpa',)  , Type : missing ,  Message: Field required



**Important Pydantic Methods to Remember**

| Method | Purpose |
|---|---|
| `e.error_count()` | Returns the number of validation errors |
| `e.errors()` | Returns detailed errors as a list |
| `err['loc']` | Identifies the field causing the error |
| `err['type']` | Identifies the error category |
| `err['msg']` | Returns the error message |


In [20]:
"""
Try writing a small Product(BaseModel) in your notebook with fields: 
product_id: int, name: str, price: float, in_stock: bool, 
and test it with:

Valid coerced inputs (product_id="50", price="199.99", in_stock="1").
A broken input inside try / except ValidationError as e: and print e.error_count().
"""

from pydantic import BaseModel, ValidationError

class Product(BaseModel):
    product_id: int
    name: str
    price: float
    in_stock: bool
    
values = {
    "product_id":"50",
    "price":"199.99",
    "in_stock":"1"
}

try:
    p1 = Product(**values)
except ValidationError as e:
    print(f"Total number of errors found: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of errors found: 1
Fields : ('name',)  , Type : missing ,  Message: Field required


> **Case A - Required and Non-nullable**

what happens when value is not provided in dict 

In [ ]:
from pydantic import BaseModel, ValidationError 

class Product(BaseModel):
    product_id: int
    name: str 

value = {
    "product_id" : "50"
}

try:
    p1 = Product(**value)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : missing ,  Message: Field required


what happens if value is None

In [16]:
class Product(BaseModel):
    product_id: int
    name: str

value = {
    "product_id" : "50",
    "name": None
}

try:
    p1 = Product(**value)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : string_type ,  Message: Input should be a valid string


> **Case B - Required and nullable** 

what happens when value is not provided in dict 

In [13]:
from pydantic import BaseModel, ValidationError 

class Product(BaseModel):
    product_id: int
    name: str | None
    
value = {
    "product_id" : "50"
}

try:
    p1 = Product(**value)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

Total number of error: 1
Fields : ('name',)  , Type : missing ,  Message: Field required


what happens if value is None

In [15]:
value = {
    "product_id" : "50",
    "name": None
}

try:
    p1 = Product(**value)
    print(p1)
except ValidationError as e:
    print(f"Total number of error: {e.error_count()}")
    for err in e.errors():
        print(f"Fields : {err['loc']}  , Type : {err['type']} ,  Message: {err['msg']}")

product_id=50 name=None


> **Case C — Optional field with default None**